In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = pd.read_excel("información_préstamos.xlsx")
X = df.drop(columns=["Impago", "ID"])
y = df["Impago"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

In [ ]:
model = LogisticRegression(
    max_iter=500,
    solver="lbfgs",
    n_jobs=-1
)

In [ ]:
pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_proba = pipeline.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)

print("AUC:", round(auc, 4))
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
df["PD_Impago"] = pipeline.predict_proba(X)[:, 1]
df[["ID", "PD_Impago"]].head()

In [ ]:
df["Decision"] = np.where(df["PD_Impago"] > 0.4, "Rechazar", "Aprobar")